## 라이브러리 호출 및 데이터 로드

In [14]:
# 라이브러리 호출
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import platform
import ast
from collections import Counter
import json

warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # Mac
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

# 게임단위

#### 컬럼명세서 

##### explode_champion_items_1

- active_synergies : 활성화된 시너지 (숫자는 실제 유저가 가지고 있는 시너지의 숫자에 해당하는 최소 활성화 수치)
   - ( 시너지를 가지고 있었지만 활성화되지 않은 시너지를 가지고 있었던 경우 {}임.)
- user_id : 폭발 후 헷갈리지않게 하기 위해 유저 아이디 파생 컬럼 생성
- flag_1 : 콤비네이션 O & 챔피언 X _ 1(TRUE) = 챔피언 없었다 / 0(FALSE) = 있었다 
- flag_2 : 콤비네이션 X & 챔피언 O _ 1(TRUE) = 콤비네이션 없었다 / 0(FALSE) = 있었다 
- top4_flag = ranked = 1~4 -> 1 (True) / 5~8 > 0 (False)
- ranked_1 = ranked = 1 -> 1 (True) / 2~8 > 0 (False)
- tier : 티어
- champion_name / name : 챔피언 이름
- star	: 별 ( 별 갯수가 많을수록 세다)
- items : 아이템 ID번호
- cost : 가격
- origin : 계열 시너지 
- class : 직업 시너지


#### 컬럼 명세서

##### explode_combination_1

- synergy : 활성화된 것과 활성화 되지 않은 시너지값들을 하나씩 꺼내 놓은 것.
- synergy_val : 실제 유저가 가지고 있는 시너지의 숫자

In [15]:
df = pd.read_csv("게임단위_게임데이터_상위랭커보존-explode_items_1.csv")

In [37]:
df.head()

,gameid,user_id,tier,ranked,ranked_1,top4_flag,champion_name,items,star,flag_1,flag_2,name,cost,origin,class,item_count,item_id,item_name
0,KR_4291707834,KR-USER-1,platinum,5,0,0,ziggs,7.0,1,0,0,ziggs,1,Rebel,Demolitionist,1,7.0,Giant's Belt
1,KR_4291707834,KR-USER-1,platinum,5,0,0,ashe,9.0,1,0,0,ashe,3,Celestial,Sniper,1,9.0,Sparring Gloves
2,KR_4291707834,KR-USER-1,platinum,5,0,0,chogath,6.0,1,0,0,chogath,4,Void,Brawler,1,6.0,Negatron Cloak
3,KR_4291707834,KR-USER-1,platinum,5,0,0,ekko,1.0,1,0,0,ekko,5,Cybernetic,Infiltrator,1,1.0,B.F. Sword
4,KR_4291707834,KR-USER-2,platinum,3,0,1,ziggs,24.0,3,0,0,ziggs,1,Rebel,Demolitionist,1,24.0,Statikk Shiv


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4861833 entries, 0 to 4861832
Data columns (total 18 columns):
 #   Column         Dtype  
---  ------         -----  
 0   gameid         str    
 1   user_id        str    
 2   tier           str    
 3   ranked         int64  
 4   ranked_1       int64  
 5   top4_flag      int64  
 6   champion_name  str    
 7   items          float64
 8   star           int64  
 9   flag_1         int64  
 10  flag_2         int64  
 11  name           str    
 12  cost           int64  
 13  origin         str    
 14  class          str    
 15  item_count     int64  
 16  item_id        float64
 17  item_name      str    
dtypes: float64(2), int64(8), str(8)
memory usage: 667.7 MB


In [17]:
df_item = df.copy()

items / item_id => int 변환

In [18]:
df_2 = pd.read_csv("게임단위_게임데이터_상위랭커보존-explode_combination_1.csv")

In [19]:
df_2.head()

,gameid,user_id,tier,ranked,top4_flag,ranked_1,flag_1,flag_2,active_synergies,synergy,synergy_val
0,KR_4291707834,KR-USER-1,platinum,5,0,0,0,0,{},Cybernetic,1
1,KR_4291707834,KR-USER-1,platinum,5,0,0,0,0,{},Demolitionist,1
2,KR_4291707834,KR-USER-1,platinum,5,0,0,0,0,{},Infiltrator,1
3,KR_4291707834,KR-USER-1,platinum,5,0,0,0,0,{},Rebel,1
4,KR_4291707834,KR-USER-1,platinum,5,0,0,0,0,{},Brawler,1


In [20]:
df_combination = df_2.copy()

In [21]:
df_item['items'].isna().sum()

np.int64(1395733)

In [22]:
df_item[['champion_name', 'items', 'item_id', 'item_name']].head(10)

,champion_name,items,item_id,item_name
0,ziggs,7.0,7.0,Giant's Belt
1,ashe,9.0,9.0,Sparring Gloves
2,chogath,6.0,6.0,Negatron Cloak
3,ekko,1.0,1.0,B.F. Sword
4,ziggs,24.0,24.0,Statikk Shiv
5,fiora,37.0,37.0,Morellonomicon
6,leona,36.0,36.0,Ionic Spark
7,leona,24.0,24.0,Statikk Shiv
8,lucian,NaN,NaN,NaN
9,vi,5.0,5.0,Chain Vest


### items / item_id int 변환. 둘 중에 하나를 날려도 되나..?

In [23]:
# item_id, items NaN → 0으로 채우고 int 변환
df_item['item_id'] = df_item['item_id'].fillna(0).astype(int)
df_item['items'] = df_item['items'].fillna(0).astype(int)

# # item_name NaN → 'none'으로 채우기
# df_item['item_name'] = df_item['item_name'].fillna('none')
# item_name NaN 처리는 Tableau 전에 하기!!

# 확인
df_item[['champion_name', 'items', 'item_id', 'item_name']].head(10)

,champion_name,items,item_id,item_name
0,ziggs,7,7,Giant's Belt
1,ashe,9,9,Sparring Gloves
2,chogath,6,6,Negatron Cloak
3,ekko,1,1,B.F. Sword
4,ziggs,24,24,Statikk Shiv
5,fiora,37,37,Morellonomicon
6,leona,36,36,Ionic Spark
7,leona,24,24,Statikk Shiv
8,lucian,0,0,NaN
9,vi,5,5,Chain Vest


### champion_name & name / items & item_id 동일 여부 확인. name과 items 컬럼 날리기

In [24]:
# champion_name vs name 동일 여부 확인
print("champion_name == name 불일치 행 수:")
print((df_item['champion_name'] != df_item['name']).sum())

# items vs item_id 동일 여부 확인
print("\nitems == item_id 불일치 행 수:")
print((df_item['items'] != df_item['item_id']).sum())

champion_name == name 불일치 행 수:
0

items == item_id 불일치 행 수:
0


In [25]:
# name, items 컬럼 제거
df_item = df_item.drop(columns=['name', 'items'])

# 확인
print(df_item.columns.tolist())

['gameid', 'user_id', 'tier', 'ranked', 'ranked_1', 'top4_flag', 'champion_name', 'star', 'flag_1', 'flag_2', 'cost', 'origin', 'class', 'item_count', 'item_id', 'item_name']


In [26]:
df_item.info()

<class 'pandas.DataFrame'>
RangeIndex: 4861833 entries, 0 to 4861832
Data columns (total 16 columns):
 #   Column         Dtype
---  ------         -----
 0   gameid         str  
 1   user_id        str  
 2   tier           str  
 3   ranked         int64
 4   ranked_1       int64
 5   top4_flag      int64
 6   champion_name  str  
 7   star           int64
 8   flag_1         int64
 9   flag_2         int64
 10  cost           int64
 11  origin         str  
 12  class          str  
 13  item_count     int64
 14  item_id        int64
 15  item_name      str  
dtypes: int64(9), str(7)
memory usage: 593.5 MB


# item 데이터셋 탐색

In [28]:
# item 데이터셋 탐색
print("=== df_item ===")
print(f"shape: {df_item.shape}")
print(f"\n--- 결측값 ---")
print(df_item.isna().sum())

print(f"\n--- 챔피언 수 ---")
print(df_item['champion_name'].nunique())


=== df_item ===
shape: (4861833, 16)

--- 결측값 ---
gameid                 0
user_id                0
tier                   0
ranked                 0
ranked_1               0
top4_flag              0
champion_name          0
star                   0
flag_1                 0
flag_2                 0
cost                   0
origin                 0
class                  0
item_count             0
item_id                0
item_name        1395737
dtype: int64

--- 챔피언 수 ---
52


In [30]:
print(f"\n--- star 분포 ---")
print(df_item['star'].value_counts().sort_index())

print(f"\n--- cost 분포 ---")
print(df_item['cost'].value_counts().sort_index())



--- star 분포 ---
star
1     770779
2    3186903
3     904151
Name: count, dtype: int64

--- cost 분포 ---
cost
1     762857
2    1008488
3    1240326
4    1260990
5     589172
Name: count, dtype: int64


In [32]:
print(f"\n--- tier 분포 ---")
print(df_item['tier'].value_counts())

print(f"\n--- ranked 분포 ---")
print(df_item['ranked'].value_counts().sort_index())


--- tier 분포 ---
tier
grand_master    986641
diamond         984816
challenger      980808
master          978566
platinum        931002
Name: count, dtype: int64

--- ranked 분포 ---
ranked
1    688660
2    681798
3    650067
4    626880
5    605654
6    577314
7    542296
8    489164
Name: count, dtype: int64


In [ ]:
print(f"\n--- item_id 분포 ---")
print(df_item['item_id'].value_counts().sort_index().head(20))


--- item_id 분포 ---
item_id
0     1395733
1       27760
2       40450
3       33196
4       43735
5       26596
6       27617
7       32664
8        6890
9       20179
11      39695
12     101035
13      51193
14      96051
15     219607
16      56887
17      46547
18      14628
19     147471
22      79316
Name: count, dtype: int64


In [36]:
print(f"\n--- origin, class 상위 5개 확인 ---")
print(df_item[['origin', 'class']].head(5))


--- origin, class 상위 5개 확인 ---
       origin          class
0       Rebel  Demolitionist
1   Celestial         Sniper
2        Void        Brawler
3  Cybernetic    Infiltrator
4       Rebel  Demolitionist


# Champion 데이터셋 탐색

In [38]:
print("=== df_combination ===")
print(f"shape: {df_combination.shape}")
print(f"\n--- 결측값 ---")
print(df_combination.isna().sum())

print(f"\n--- synergy 수 ---")
print(df_combination['synergy'].nunique())

=== df_combination ===
shape: (3361555, 11)

--- 결측값 ---
gameid              0
user_id             0
tier                0
ranked              0
top4_flag           0
ranked_1            0
flag_1              0
flag_2              0
active_synergies    0
synergy             0
synergy_val         0
dtype: int64

--- synergy 수 ---
23


In [39]:
print(f"\n--- synergy_val 분포 ---")
print(df_combination['synergy_val'].value_counts().sort_index())

print(f"\n--- tier 분포 ---")
print(df_combination['tier'].value_counts())


--- synergy_val 분포 ---
synergy_val
0        4875
1     1579227
2      990323
3      304161
4      356956
5       19763
6       98182
7        6824
8        1028
9         215
10          1
Name: count, dtype: int64

--- tier 분포 ---
tier
diamond         689813
grand_master    681997
master          671438
challenger      670196
platinum        648111
Name: count, dtype: int64


In [42]:
print(f"\n--- ranked 분포 ---")
print(df_combination['ranked'].value_counts().sort_index())

print(f"\n--- active_synergies top 10 ---")
print(df_combination['active_synergies'].value_counts().head(10))


--- ranked 분포 ---
ranked
1    447526
2    439564
3    432510
4    425883
5    418824
6    411251
7    400980
8    385017
Name: count, dtype: int64

--- active_synergies top 10 ---
active_synergies
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Brawler': 4}                                                        217955
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Rebel': 3, 'Brawler': 4, 'Starship': 1}                              93505
{'Blaster': 2, 'Chrono': 4, 'ManaReaver': 2, 'Mercenary': 1, 'Blademaster': 3, 'Celestial': 2, 'Valkyrie': 2}     64329
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Rebel': 3, 'Brawler': 4}                                             56554
{'Blaster': 4, 'Chrono': 2, 'Brawler': 4}                                                                         51685
{'DarkStar': 3, 'Celestial': 2, 'Mystic': 2, 'Sniper': 2, 'Vanguard': 4}                                          50359
{'Mystic': 2, 'Sorcerer': 4, 'StarGuardian': 6}                                   

In [44]:
print(f"전체 조합 종류 수: {df_combination['active_synergies'].nunique()}")
print(f"\n--- active_synergies top 10 ---")
print(df_combination['active_synergies'].value_counts().head(10))

전체 조합 종류 수: 16587

--- active_synergies top 10 ---
active_synergies
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Brawler': 4}                                                        217955
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Rebel': 3, 'Brawler': 4, 'Starship': 1}                              93505
{'Blaster': 2, 'Chrono': 4, 'ManaReaver': 2, 'Mercenary': 1, 'Blademaster': 3, 'Celestial': 2, 'Valkyrie': 2}     64329
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Rebel': 3, 'Brawler': 4}                                             56554
{'Blaster': 4, 'Chrono': 2, 'Brawler': 4}                                                                         51685
{'DarkStar': 3, 'Celestial': 2, 'Mystic': 2, 'Sniper': 2, 'Vanguard': 4}                                          50359
{'Mystic': 2, 'Sorcerer': 4, 'StarGuardian': 6}                                                                   48208
{'Chrono': 2, 'Infiltrator': 2, 'Brawler': 4, 'Sorcerer': 2, 'Void': 3}                     

### 하은님께서 유저 단위의 데이터셋으로 보신 조합 픽률 게임단위에서도 확인

In [48]:
# user_id 기준 중복 제거 후 조합 픽률
print(f"\n--- user_id 기준 중복 제거 후 조합 픽률 ---")
df_combo_dedup = df_combination.drop_duplicates(subset=['user_id'])
top_combos = df_combo_dedup['active_synergies'].value_counts(normalize=True).head(5)
print(top_combos.map(lambda x: f'{x:.2%}'))


--- user_id 기준 중복 제거 후 조합 픽률 ---
active_synergies
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Brawler': 4}                                                        6.13%
{'Blaster': 4, 'Chrono': 2, 'Mercenary': 1, 'Rebel': 3, 'Brawler': 4, 'Starship': 1}                             2.35%
{'Blaster': 4, 'Chrono': 2, 'Brawler': 4}                                                                        1.82%
{'Blaster': 2, 'Chrono': 4, 'ManaReaver': 2, 'Mercenary': 1, 'Blademaster': 3, 'Celestial': 2, 'Valkyrie': 2}    1.79%
{'Mystic': 2, 'Sorcerer': 4, 'StarGuardian': 6}                                                                  1.76%
Name: proportion, dtype: str


### synergy_val 이상치 값 확인

In [53]:
result = df_combination[df_combination['synergy_val']==10]
result

,gameid,user_id,tier,ranked,top4_flag,ranked_1,flag_1,flag_2,active_synergies,synergy,synergy_val
3117462,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",DarkStar,10


In [55]:
test = df_combination[df_combination['user_id']=='KR-USER-371183']
test

,gameid,user_id,tier,ranked,top4_flag,ranked_1,flag_1,flag_2,active_synergies,synergy,synergy_val
3117461,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",Chrono,1
3117462,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",DarkStar,10
3117463,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",Infiltrator,2
3117464,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",ManaReaver,1
3117465,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",Protector,1
3117466,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",Celestial,2
3117467,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",Mystic,2
3117468,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",Sorcerer,2
3117469,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",Sniper,2
3117470,KR_4338954957,KR-USER-371183,challenger,1,1,1,0,0,"{'DarkStar': 9, 'Infiltrator': 2, 'Celestial': 2, 'Mystic': 2, 'Sorcerer': 2, 'Sniper': 2}",Vanguard,1
